# SpendRadar · Moirai 2.0 (Salesforce) — 팀원 데이터 일괄 실행

팀원 3명(U1·U2·U4, 포함 조건은 Baseline 레포 data/data_inclusion_by_user.csv)의 월간 지출로 사람마다 다음 두 가지를 만듭니다.
1. **검증**: 마지막 6개월(2026-05 ~ 2026-10)을 가리고 그 이전 데이터로 예측 → 3대 지표(MAE · MASE · WQL)
2. **향후 6개월 예측**(2026-11 ~ 2027-04): 재귀 방식(1개월 예측 분포에서 샘플 경로 200개를 만들어 6번 반복 → 6개월 불확실성이 누적됨)

**입력 데이터**: 저장소의 `data/team_monthly_long.csv` (팀원 3명 U1·U2·U4, 카테고리 6개 + 전체) — 2번 셀이 자동으로 불러옴 (Baseline 레포와 같은 파일)

**사용법**: 런타임 → 런타임 유형 변경 → **GPU(T4)** → 위에서부터 셀을 하나씩 차례로 실행

**출력**: 사람마다 표 4개 · 그래프 3개 (카테고리 6개 + 전체), 마지막에 `ALL_metrics_moirai.csv` · `ALL_forecast_moirai.csv` · `ALL_validation_moirai.csv`

In [ ]:
# 1. 설치 + 확인
MODELS = ["moirai"]
STRATEGY = "recursive"   # recursive = 재귀(샘플 경로) / direct = 한 번에 6개월 / both = 둘 다 / recursive_median = 이전 방식(중앙값만 붙임)
TAG = "moirai"   # 결과 파일 이름에 붙는 이름

# uni2ts는 torch<2.5·numpy 1.26으로 묶여 Colab과 충돌 → 본체만 설치 + 필요한 부품만 설치
!pip install -q --no-deps "uni2ts @ git+https://github.com/SalesforceAIResearch/uni2ts.git"
!pip install -q gluonts einops jaxtyping hydra-core lightning python-dotenv orjson multiprocess openpyxl
!apt-get -qq install -y fonts-nanum > /dev/null   # 그래프 한글 글꼴

import importlib; importlib.invalidate_caches()
try:
    importlib.import_module("uni2ts.model.moirai2")
    print("✅ moirai 설치 확인 — 다음 셀로 진행하세요")
except Exception as e:
    print("❌ moirai 설치 실패:", type(e).__name__, e)
    raise

In [ ]:
# 2. 데이터 불러오기: 저장소의 data/team_monthly_long.csv (팀원 3명 U1·U2·U4, 카테고리 6개 + 전체)
REPO_URL = "https://github.com/spendRadar/spendRadar-TeamData-FM.git"   # 저장소 주소가 다르면 수정
import os
!test -d /content/repo || git clone -q $REPO_URL /content/repo
DATA = "/content/repo/data/team_monthly_long.csv"
if not os.path.exists(DATA):            # 저장소를 못 받으면 CSV를 직접 업로드
    from google.colab import files
    DATA = next(iter(files.upload()))
print("데이터:", DATA)

In [ ]:
# 3. 공통 설정 + 데이터 정리 (보통 바꿀 필요 없음)
import time, numpy as np, pandas as pd
CATEGORIES = ['식비 / 카페', '주거 / 통신', '쇼핑 / 패션', '교통 / 차량', '문화 / 여가', '의료']
SERIES = CATEGORIES + ['전체']   # 전체 = 6개 카테고리 합계, 별도 시계열로 예측
HORIZON = 6      # 검증·예측 길이 (개월). split = test인 마지막 6개월을 가리고 맞힘
CHUNK = 1        # 재귀 예측에서 한 번에 예측할 개월 수

def load_team(path):
    '''long 형식 CSV → {사용자 ID: dict(user, monthly, series, end, test_start)} (src/spendradar_team.py와 같음)'''
    df = pd.read_csv(path, parse_dates=["ds"])
    people = {}
    for uid, g in df.groupby("entity_id", sort=True):
        wide = g.pivot(index="ds", columns="category", values="y")
        wide.index = pd.PeriodIndex(wide.index, freq="M")
        monthly = wide.reindex(columns=SERIES).astype(float)
        test = g[g.split == "test"].ds
        people[uid] = dict(user=uid, monthly=monthly,
                           series={c: monthly[c].astype("float32") for c in SERIES},
                           end=str(monthly.index[-1]), test_start=str(test.min().to_period("M")))
        print(f"{uid}: {monthly.index[0]} ~ {monthly.index[-1]} ({len(monthly)}개월) · 검증 {people[uid]['test_start']}부터 {HORIZON}개월 · "
              f"예측 {monthly.index[-1] + 1}부터 {HORIZON}개월")
    return people

PEOPLE = load_team(DATA)
print("\n준비된 사용자:", list(PEOPLE))

In [ ]:
# 4. 모델 불러오기: Moirai 2.0 (Salesforce)
try:
    import torch; DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
except ImportError:
    DEVICE = "cpu"
Q = tuple(round(0.1 * k, 1) for k in range(1, 10))     # 분위수 10%~90% (WQL용 9개)
PRED = {}

# ── Moirai 2.0 (gluonts 예측기 대신 predict 직접 호출) ──
if "moirai" in MODELS:
    from uni2ts.model.moirai2 import Moirai2Forecast, Moirai2Module
    _moi = Moirai2Module.from_pretrained("Salesforce/moirai-2.0-R-small")
    _QL = np.asarray(_moi.quantile_levels, float)
    def _moirai(ctxs, h):
        m = Moirai2Forecast(module=_moi, prediction_length=h, context_length=max(map(len, ctxs)),
                            target_dim=1, feat_dynamic_real_dim=0, past_feat_dynamic_real_dim=0).to(DEVICE)
        out = np.asarray(m.predict(past_target=[np.asarray(c, dtype=np.float32) for c in ctxs]))
        if out.ndim == 4: out = out[..., 0]
        return {q: out[:, int(np.argmin(np.abs(_QL - q))), :h] for q in Q}
    PRED["moirai"] = _moirai


N_PATHS = 200    # 재귀 예측 샘플 경로 수 (층화 샘플링 기준 200개면 시드에 따른 MAE 차이 0.2% 이내)
SEED = 0         # 샘플링 난수 시드 (같은 값이면 같은 결과)


def _draw(qv, u):
    """1개월 예측 분포(분위수 0.1 ~ 0.9)에서 확률 u에 해당하는 값 (역분포 샘플링, 행마다 1개)
    분위수 사이는 선형 보간, 양 끝 10%는 가장 가까운 구간의 기울기로 연장, 지출이라 0 미만은 0"""
    lv = np.array(Q)
    j = np.clip(np.searchsorted(lv, u) - 1, 0, len(Q) - 2)
    r = np.arange(len(qv))
    x0, x1 = qv[r, j], qv[r, j + 1]
    return np.maximum(x0 + (u - lv[j]) / 0.1 * (x1 - x0), 0)


def forecast(model, ctxs, h, strategy=STRATEGY, chunk=CHUNK):
    """direct    : 한 번에 h개월 예측
    recursive : 샘플 경로 재귀 — 1개월 예측 분포에서 경로마다 값을 하나씩 뽑아(층화 샘플링) 그 경로의 과거 뒤에 붙임 → h번 반복
                → 경로 N_PATHS개의 월별 분포에서 분위수 계산. 앞 달의 불확실성이 다음 달 입력에 들어가 누적됨
    recursive_median : (이전 방식, 비교용) 1개월 예측의 중앙값만 붙임 → 각 달 범위는 그 한 달의 불확실성만 반영
    전 기간 0원인 카테고리는 모델에 넣지 않고 0원 예측, 음수는 0으로 자르고 분위수는 크기순 정렬"""
    def f(cs, k):
        live = [i for i, c in enumerate(cs) if np.any(np.asarray(c) != 0)]
        res = {q: np.zeros((len(cs), k)) for q in Q}
        if live:
            p = PRED[model]([cs[i] for i in live], k)
            for q in Q: res[q][live] = np.asarray(p[q])[:, :k]
        a = np.sort(np.stack([np.maximum(res[q], 0) for q in Q]), axis=0)
        return {q: a[j] for j, q in enumerate(Q)}
    if strategy == "direct":
        return f(ctxs, h)
    if strategy == "recursive_median":
        cs = [np.asarray(c, dtype=np.float32).copy() for c in ctxs]
        out = {q: np.zeros((len(cs), h)) for q in Q}; done = 0
        while done < h:
            k = min(chunk, h - done); p = f(cs, k)
            for q in Q: out[q][:, done:done + k] = p[q][:, :k]
            cs = [np.concatenate([c, p[0.5][i, :k].astype(np.float32)]) for i, c in enumerate(cs)]
            done += k
        return out
    # recursive (샘플 경로): 시계열 S개 × 경로 N_PATHS개를 한 번에 모델에 넣음
    rng = np.random.default_rng(SEED)
    S = len(ctxs)
    paths = [np.asarray(c, dtype=np.float32) for c in ctxs for _ in range(N_PATHS)]
    sims = np.zeros((S, N_PATHS, h))
    for t in range(h):
        p = f(paths, 1)
        # 층화 샘플링: 시계열마다 경로 N개에 확률 구간 [0,1)을 N등분해 하나씩 배정 (무작위 순서) → 적은 경로로도 결과가 안정적
        u = ((rng.permuted(np.tile(np.arange(N_PATHS), (S, 1)), axis=1) + rng.random((S, N_PATHS))) / N_PATHS).ravel()
        x = _draw(np.stack([p[q][:, 0] for q in Q], axis=1), u)
        sims[:, :, t] = x.reshape(S, N_PATHS)
        paths = [np.concatenate([c, np.float32([v])]) for c, v in zip(paths, x)]
    return {q: np.quantile(sims, q, axis=1) for q in Q}

print("준비된 모델:", list(PRED), "· 예측 방식:", {"recursive": f"재귀 (샘플 경로 {N_PATHS}개)", "direct": "직접", "both": f"직접 + 재귀 (샘플 경로 {N_PATHS}개)", "recursive_median": "재귀 (중앙값, 이전 방식)"}[STRATEGY])

In [ ]:
# 5. 일괄 실행: 사람 × 모델마다 3대 지표 검증 + 향후 6개월 예측
def evaluate(model, d, strategy):
    """사용자 한 명: 검증(test) 지표 met · 향후 6개월 예측 fc · 검증 실제 vs 예측 val
    시계열 = 카테고리 6개 + 전체(합계), 각각 따로 예측
    MAE  = 월 평균 |실제 − 예측 중앙값| (원/월)
    MASE = MAE ÷ 입력 구간 '지난달 값' 예측의 평균 오차 (lag 1)
    WQL  = 2 × Σpinball ÷ Σ|실제| (분위수 9개 평균)
    '종합' 행 = 카테고리 6개 기준 (MAE 합 · MASE 평균 · WQL 합쳐서 계산 · 적중률 평균), 전체 시계열은 포함하지 않음
    지출이 한 번도 없는 시계열은 MASE·WQL·적중률에서 제외"""
    series, monthly = d["series"], d["monthly"]
    t0 = pd.Period(d["test_start"], "M"); tm = pd.period_range(t0, periods=HORIZON, freq="M")
    ctxs = [series[c][series[c].index < t0].values for c in SERIES]
    actual = np.stack([series[c][tm].values for c in SERIES]).astype(float)
    nz = [c for c in SERIES if monthly[c].sum() > 0]
    P = forecast(model, ctxs, HORIZON, strategy)
    rows, pin_all, y_all = [], 0.0, 0.0
    for i, c in enumerate(SERIES):
        y, med = actual[i], P[0.5][i]
        mae = np.abs(y - med).mean()
        ctx = np.asarray(ctxs[i], float); sc = np.mean(np.abs(np.diff(ctx))) if len(ctx) > 1 else np.nan
        pin = sum(np.maximum(q * (y - P[q][i]), (q - 1) * (y - P[q][i])).sum() for q in Q) / len(Q)
        ys = np.abs(y).sum()
        if c in nz and c in CATEGORIES: pin_all += pin; y_all += ys
        rows.append({"카테고리": c, "MAE(원/월)": mae, "MASE": mae / sc if c in nz and sc > 0 else np.nan,
                     "WQL": 2 * pin / ys if ys > 0 else np.nan,
                     "예상범위 적중률": ((y >= P[0.1][i]) & (y <= P[0.9][i])).mean() if c in nz else np.nan})
    val = pd.DataFrame([{"카테고리": c, "월": str(m), "실제": actual[i, h], "예측": P[0.5][i, h],
                         "하한(10%)": P[0.1][i, h], "상한(90%)": P[0.9][i, h]}
                        for i, c in enumerate(SERIES) for h, m in enumerate(tm)])
    met = pd.DataFrame(rows).set_index("카테고리")
    cat = met.loc[CATEGORIES]
    met.loc["종합"] = {"MAE(원/월)": cat["MAE(원/월)"].sum(), "MASE": cat["MASE"].mean(),
                      "WQL": 2 * pin_all / y_all, "예상범위 적중률": cat["예상범위 적중률"].mean()}
    # 향후 6개월 (전체 기간을 입력으로)
    fut = pd.period_range(pd.Period(d["end"], "M") + 1, periods=HORIZON, freq="M")
    F = forecast(model, [series[c].values for c in SERIES], HORIZON, strategy)
    fc = pd.DataFrame([{"카테고리": c, "월": str(m), "예측": round(F[0.5][i, h]),
                        "하한(10%)": round(F[0.1][i, h]), "상한(90%)": round(F[0.9][i, h])}
                       for i, c in enumerate(SERIES) for h, m in enumerate(fut)])
    return met, fc, val

STRATS = ["direct", "recursive"] if STRATEGY == "both" else [STRATEGY]
label = lambda m, s: f"{m}({'재귀' if s == 'recursive' else '직접'})"
ALL_M, ALL_F, ALL_V, RESULTS = [], [], [], {}
for user, d in PEOPLE.items():
    for m in PRED:
        for st in STRATS:
            t = time.time()
            try:
                met, fc, val = evaluate(m, d, st)
            except Exception as e:
                print(f"❌ {user} · {label(m, st)}: {type(e).__name__}: {e}"); continue
            r = met.loc["종합"]
            print(f"✅ {user:4s} · {label(m, st):14s} MAE {r['MAE(원/월)']:>11,.0f}  MASE {r['MASE']:.3f}  WQL {r['WQL']:.3f}  "
                  f"적중 {r['예상범위 적중률']:.0%}  · 6개월 예측(전체) {fc[fc['카테고리'] == '전체']['예측'].sum():>11,.0f}원  ({time.time()-t:.1f}초)")
            mo = met.reset_index(); mo.insert(0, "모델", label(m, st)); mo.insert(1, "사용자", user)
            fo = fc.copy();         fo.insert(0, "모델", label(m, st)); fo.insert(1, "사용자", user)
            vo = val.copy();        vo.insert(0, "모델", label(m, st)); vo.insert(1, "사용자", user)
            ALL_M.append(mo); ALL_F.append(fo); ALL_V.append(vo)
            RESULTS[(user, label(m, st))] = dict(met=met, fc=fc, val=val, d=d)

ALL_M = pd.concat(ALL_M, ignore_index=True); ALL_F = pd.concat(ALL_F, ignore_index=True)
ALL_V = pd.concat(ALL_V, ignore_index=True)
print("\n완료:", len(RESULTS), "개 (사람 × 모델) — 6번 셀에서 사람별 결과, 7번 셀에서 요약·CSV 저장")

In [ ]:
# 6. 사람별 상세 결과 (표 4개 · 그래프 3개): 3대 지표 · 검증 실제 vs 예측 · 6개월 예측 (카테고리 6개 + 전체)
import matplotlib.pyplot as plt
from matplotlib import font_manager, rcParams
for _f in font_manager.findSystemFonts():
    if "NanumGothic.ttf" in _f:
        font_manager.fontManager.addfont(_f); rcParams["font.family"] = "NanumGothic"
rcParams["axes.unicode_minus"] = False
SHOW_GRAPHS = True     # False면 표만

def show_person(user, lab):
    r = RESULTS[(user, lab)]; met, fc, val, d = r["met"], r["fc"], r["val"], r["d"]
    series, monthly = d["series"], d["monthly"]; CATS = SERIES
    nz = [c for c in CATS if monthly[c].sum() > 0]
    o = met.loc["종합"]
    print("\n" + "━" * 90)
    print(f"■ {user} · {lab} · 검증 {val['월'].min()}~{val['월'].max()}")
    print(f"  MAE {o['MAE(원/월)']:>12,.0f} 원/월 (카테고리 6개 합) · MASE {o['MASE']:.3f} · WQL {o['WQL']:.3f} · "
          f"예상 범위 적중률 {o['예상범위 적중률']:.0%}")
    print("━" * 90)

    print(f"📋 표 1. [{user}] 카테고리별 3대 지표")
    display(met.style.format({"MAE(원/월)": "{:,.0f}", "MASE": "{:.3f}", "WQL": "{:.3f}", "예상범위 적중률": "{:.0%}"}, na_rep="-")
            .set_properties(subset=pd.IndexSlice[["전체", "종합"], :], **{"font-weight": "bold", "background-color": "#eef4ff"}))

    cmp_tbl = val.melt(id_vars=["카테고리", "월"], value_vars=["실제", "예측"], var_name="구분").pivot_table(
        index=["카테고리", "구분"], columns="월", values="value").reindex(
        pd.MultiIndex.from_product([CATS, ["실제", "예측"]]))
    print(f"\n📋 표 2. [{user}] 검증 구간 실제 vs 예측 (원)")
    display(cmp_tbl.style.format("{:,.0f}"))

    tb = fc.pivot(index="카테고리", columns="월", values="예측").reindex(CATS)
    tb.loc["카테고리 합"] = tb.loc[CATEGORIES].sum(); tb["6개월 합계"] = tb.sum(axis=1)
    print(f"\n📋 표 3. [{user}] 향후 6개월 카테고리별 예측 (원)")
    display(tb.style.format("{:,.0f}")
            .background_gradient(cmap="Blues", subset=pd.IndexSlice[CATEGORIES, list(tb.columns[:-1])], axis=None)
            .set_properties(subset=pd.IndexSlice[["전체", "카테고리 합"], :], **{"font-weight": "bold"}))
    rg = fc.assign(범위=lambda x: x.apply(lambda q: f"{q['하한(10%)']/1e4:,.0f} ~ {q['상한(90%)']/1e4:,.0f}만", axis=1))
    print(f"\n📋 표 4. [{user}] 예상 범위 (10~90%)")
    display(rg.pivot(index="카테고리", columns="월", values="범위").reindex(CATS))

    if not SHOW_GRAPHS: return
    # 그래프 1: 검증 구간
    fig, axes = plt.subplots(2, 4, figsize=(19, 7.2), squeeze=False)
    for ax, c in zip(axes.flat, CATS):
        v = val[val["카테고리"] == c]; xt = pd.PeriodIndex(v["월"], freq="M").to_timestamp(); h = series[c]
        ax.plot(h.index.to_timestamp(), h.values / 1e4, color="black", lw=1.5, label="실제")
        ax.axvspan(xt[0], xt[-1], color="#f1f3f5", zorder=0)
        ax.plot(xt, v["예측"] / 1e4, "--o", ms=3, lw=1.8, color="#1f6feb", label="예측")
        ax.fill_between(xt, v["하한(10%)"] / 1e4, v["상한(90%)"] / 1e4, color="#1f6feb", alpha=.15, label="예상 범위")
        m_ = met.loc[c]
        ax.set_title(c + ("" if c not in nz else f"\nMAE {m_['MAE(원/월)']/1e4:.1f}만 · MASE {m_['MASE']:.2f} · WQL {m_['WQL']:.2f}"), fontsize=9)
        ax.set_ylabel("만 원"); ax.tick_params(axis="x", labelrotation=45, labelsize=8)
    axes.flat[0].legend(fontsize=8); axes.flat[-1].axis("off")
    fig.suptitle(f"그래프 1. [{user}] {lab} 검증 — MAE {o['MAE(원/월)']/1e4:,.1f}만 · MASE {o['MASE']:.3f} · WQL {o['WQL']:.3f} (회색 = 가린 6개월)", fontsize=12)
    plt.tight_layout(); plt.show()
    # 그래프 2: 향후 6개월
    fig, axes = plt.subplots(2, 4, figsize=(19, 7.2), squeeze=False)
    for ax, c in zip(axes.flat, CATS):
        f_ = fc[fc["카테고리"] == c]; x = pd.PeriodIndex(f_["월"], freq="M").to_timestamp(); h = series[c]
        ax.plot(h.index.to_timestamp(), h.values / 1e4, color="#555", lw=1.5, label="실제")
        ax.plot(x, f_["예측"] / 1e4, "--o", ms=3, lw=1.8, color="#1f6feb", label="예측")
        ax.fill_between(x, f_["하한(10%)"] / 1e4, f_["상한(90%)"] / 1e4, color="#1f6feb", alpha=.18, label="예상 범위")
        ax.set_title(f"{c}  (6개월 {f_['예측'].sum()/1e4:,.0f}만 원)", fontsize=10)
        ax.set_ylabel("만 원"); ax.tick_params(axis="x", labelrotation=45, labelsize=8)
    axes.flat[0].legend(fontsize=8); axes.flat[-1].axis("off")
    fig.suptitle(f"그래프 2. [{user}] {lab} 향후 6개월 카테고리별 예측", fontsize=12)
    plt.tight_layout(); plt.show()
    # 그래프 3: 월 총지출
    fig, ax = plt.subplots(figsize=(16, 4.2))
    tot = monthly["전체"]; ft = fc[fc["카테고리"] == "전체"].set_index("월")["예측"]
    ax.bar([str(m) for m in tot.index], tot.values / 1e4, color="#adb5bd", label="실제 월 총지출")
    ax.bar(ft.index, ft.values / 1e4, color="#1f6feb", label="예측 월 총지출")
    for k, v in enumerate(ft.values):
        ax.text(len(tot) + k, v / 1e4, f"{v/1e4:,.0f}", ha="center", va="bottom", fontsize=8, color="#1f6feb")
    ax.tick_params(axis="x", labelrotation=45, labelsize=8); ax.set_ylabel("만 원"); ax.legend(); ax.grid(axis="y", alpha=.3)
    ax.set_title(f"그래프 3. [{user}] 월 총지출 — 실제 + {lab} 예측 (6개월 합계 {ft.sum()/1e4:,.0f}만 원)")
    plt.tight_layout(); plt.show()

for (user, lab) in RESULTS:
    show_person(user, lab)

In [ ]:
# 7. 결과 요약: 사람별 3대 지표 + 6개월 예측 합계 → 결과 CSV 3개 저장·다운로드
from google.colab import files
tot = ALL_M[ALL_M["카테고리"] == "종합"].set_index(["사용자", "모델"])[["MAE(원/월)", "MASE", "WQL", "예상범위 적중률"]]
tot["6개월 예측 합계(원, 전체)"] = ALL_F[ALL_F["카테고리"] == "전체"].groupby(["사용자", "모델"])["예측"].sum()
print("📋 사람별 3대 지표 (낮을수록 좋음 · MASE 1 미만 = '지난달 값' 단순 예측보다 나음)")
display(tot.style.format({"MAE(원/월)": "{:,.0f}", "MASE": "{:.3f}", "WQL": "{:.3f}",
                          "예상범위 적중률": "{:.0%}", "6개월 예측 합계(원, 전체)": "{:,.0f}"}))
print("\n📋 카테고리별 MAE (원/월)")
display(ALL_M.pivot_table(index=["사용자", "모델"], columns="카테고리", values="MAE(원/월)")[SERIES + ["종합"]]
        .style.format("{:,.0f}", na_rep="-"))
ALL_M.to_csv(f"ALL_metrics_{TAG}.csv", index=False, encoding="utf-8-sig")
ALL_F.to_csv(f"ALL_forecast_{TAG}.csv", index=False, encoding="utf-8-sig")
ALL_V.to_csv(f"ALL_validation_{TAG}.csv", index=False, encoding="utf-8-sig")
files.download(f"ALL_metrics_{TAG}.csv"); files.download(f"ALL_forecast_{TAG}.csv"); files.download(f"ALL_validation_{TAG}.csv")
print(f"\n⬇ ALL_metrics_{TAG}.csv · ALL_forecast_{TAG}.csv · ALL_validation_{TAG}.csv 저장 완료 → results/{TAG}/ 에 넣으면 분석 코드가 읽습니다")